In [ ]:
from pathlib import Path
import os
import sys

project_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file())
sys.path.insert(0, str(project_root))
from experiments.paths import GENERATED, PROJECT_ROOT

run_directory = GENERATED / 'soay/age_size/training/transformer'
run_directory.mkdir(parents=True, exist_ok=True)
os.chdir(run_directory)


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D
from scipy.stats import norm

current_dir = Path.cwd().resolve()
repo_root = PROJECT_ROOT

data_path = GENERATED / 'soay/age_size/data/gp_ibm_dataset_agesize.csv'
samples_path = GENERATED / 'soay/age_size/inference/gpdf_agesize_gp_samples.npz'

soay_data = pd.read_csv(data_path)

with np.load(samples_path) as saved:
    pooled_lengthscales = np.concatenate([saved["growth_ls"], saved["surv_ls"], saved["repr_ls"]], axis=0)

size_data = soay_data.dropna(subset=["z", "age"]).copy()
size_data["age"] = size_data["age"].astype(int)

plt.style.use("seaborn-v0_8-white")

def style_axis(axis):
    axis.tick_params(labelsize=9, direction="out")
    axis.grid(False)
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)

In [ ]:
ages = np.sort(size_data["age"].unique())
age_colors = dict(zip(ages, sns.color_palette("viridis", n_colors=len(ages))))

figure, axis = plt.subplots(figsize=(7.2, 3.4), constrained_layout=True)

sns.kdeplot(
    data=size_data,
    x="z",
    hue="age",
    hue_order=ages,
    palette=age_colors,
    common_norm=False,
    cut=0,
    linewidth=1.4,
    alpha=0.85,
    ax=axis,
)

axis.set_xlabel("Size", fontsize=14)
axis.set_ylabel("Density", fontsize=14)
axis.set_title("Size Distribution By Age Across All Years", fontsize=16)
axis.tick_params(labelsize=9, direction="out")
axis.grid(False)
axis.spines["top"].set_visible(False)
axis.spines["right"].set_visible(False)

sns.move_legend(
    axis,
    "center left",
    bbox_to_anchor=(1.02, 0.5),
    title="Age",
    frameon=False,
)

figure.savefig("size_distribution_by_age_all_years.pdf", bbox_inches="tight")
figure.savefig("size_distribution_by_age_all_years.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
size_data["age_group"] = np.select(
    [size_data["age"] == 0, size_data["age"] == 1, size_data["age"] == 2],
    ["Age 0", "Age 1", "Age 2"],
    default="Age 3+",
)

age_groups = ["Age 0", "Age 1", "Age 2", "Age 3+"]
colors = {"Age 0": "#0072B2", "Age 1": "#D55E00", "Age 2": "#009E73", "Age 3+": "#CC79A7"}

# Separate normal training priors: group -> (mean, standard deviation)
prior_parameters = {
    "Age 0": (2.6, 0.2),
    "Age 1": (2.9, 0.2),
    "Age 2": (3.0, 0.2),
    "Age 3+": (3.15, 0.20),
}

prior_lower = min(mean - 4 * sd for mean, sd in prior_parameters.values())
prior_upper = max(mean + 4 * sd for mean, sd in prior_parameters.values())
size_lower = min(size_data["z"].min(), prior_lower)
size_upper = max(size_data["z"].max(), prior_upper)
size_grid = np.linspace(size_lower, size_upper, 500)

figure, axes = plt.subplots(2, 2, figsize=(7.2, 6), sharex=True, sharey=True)
for axis, age_group in zip(axes.ravel(), age_groups):
    values = size_data.loc[size_data["age_group"] == age_group, "z"]
    prior_mean, prior_sd = prior_parameters[age_group]
    prior_density = norm.pdf(size_grid, loc=prior_mean, scale=prior_sd)

    sns.kdeplot(x=values, fill=True, cut=0, color=colors[age_group], linewidth=0, alpha=0.2, ax=axis)
    sns.kdeplot(x=values, cut=0, color=colors[age_group], linewidth=1.8, ax=axis)
    axis.plot(size_grid, prior_density, color="#333333", linestyle="--", linewidth=1.8)

    axis.set_title(age_group, fontsize=11)
    axis.set_xlim(size_lower, size_upper)
    axis.set_xlabel("")
    axis.set_ylabel("")
    axis.text(
        0.04,
        0.94,
        rf"Prior: $\mathcal{{N}}({prior_mean:.2f},{prior_sd:.2f}^2)$",
        transform=axis.transAxes,
        va="top",
        fontsize=9,
    )
    style_axis(axis)

legend_handles = [
    Line2D([0], [0], color="#555555", linewidth=2, label="Observed Density"),
    Line2D([0], [0], color="#333333", linestyle="--", linewidth=2, label="Training Prior"),
]

figure.subplots_adjust(left=0.10, right=0.97, bottom=0.10, top=0.80, hspace=0.25, wspace=0.12)

figure.suptitle("Age-Specific Size Distributions And Training Priors", fontsize=16, y=0.97)
figure.legend(handles=legend_handles, loc="upper center", bbox_to_anchor=(0.5, 0.91), ncol=2, frameon=False)
figure.supxlabel("Size", fontsize=14)
figure.supylabel("Density", fontsize=14)

figure.savefig("age_specific_size_priors.pdf", bbox_inches="tight")
figure.savefig("age_specific_size_priors.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
age_observations = size_data["age"]

figure, axis = plt.subplots(figsize=(7.2, 4.0), constrained_layout=True)

sns.histplot(
    x=age_observations,
    discrete=True,
    stat="probability",
    shrink=0.8,
    color="#D55E00",
    alpha=0.85,
    ax=axis,
)

axis.set_xticks(np.arange(9))
axis.set_xlabel("Age", fontsize=14)
axis.set_ylabel("Proportion", fontsize=14)
axis.set_title("Age Distribution Across All Years", fontsize=16)
axis.tick_params(labelsize=9, direction="out")
axis.grid(False)
axis.spines["top"].set_visible(False)
axis.spines["right"].set_visible(False)

figure.savefig("age_distribution.pdf", bbox_inches="tight")
figure.savefig("age_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
size_lengthscales = pooled_lengthscales[:, 0]
age_lengthscales = pooled_lengthscales[:, 1]

size_prior_log_mean = 0.0
size_prior_log_sd = 1.5
age_prior_log_mean = 2.0
age_prior_log_sd = 1.5

log_grid = np.linspace(-5.5, 5.5, 1000)
size_prior_density = norm.pdf(log_grid, loc=size_prior_log_mean, scale=size_prior_log_sd)
age_prior_density = norm.pdf(log_grid, loc=age_prior_log_mean, scale=age_prior_log_sd)

figure, axes = plt.subplots(1, 2, figsize=(7.2, 3.4), sharex=True, constrained_layout=True)

sns.kdeplot(x=np.log(size_lengthscales), fill=True, cut=0, color="#0072B2", linewidth=0, alpha=0.2, ax=axes[0])
sns.kdeplot(x=np.log(size_lengthscales), cut=0, color="#0072B2", linewidth=1.8, ax=axes[0])
axes[0].plot(log_grid, size_prior_density, color="#333333", linestyle="--", linewidth=1.8)
axes[0].set_title("Size Lengthscale", fontsize=11)

sns.kdeplot(x=np.log(age_lengthscales), fill=True, cut=0, color="#D55E00", linewidth=0, alpha=0.2, ax=axes[1])
sns.kdeplot(x=np.log(age_lengthscales), cut=0, color="#D55E00", linewidth=1.8, ax=axes[1])
axes[1].plot(log_grid, age_prior_density, color="#333333", linestyle="--", linewidth=1.8)
axes[1].set_title("Age Lengthscale", fontsize=11)

for axis in axes:
    axis.set_xlim(-5.5, 5.5)
    axis.set_xlabel("")
    axis.set_ylabel("")
    style_axis(axis)

legend_handles = [
    Line2D([0], [0], color="#0072B2", linewidth=2, label="Size Posterior"),
    Line2D([0], [0], color="#333333", linestyle="--", linewidth=2, label=r"Size Prior: $\sim \mathcal{N}(0,1.5^2)$"),
    Line2D([0], [0], color="#D55E00", linewidth=2, label="Age Posterior"),
    Line2D([0], [0], color="#333333", linestyle="--", linewidth=2, label=r"Age Prior: $\sim \mathcal{N}(2,1.5^2)$"),
]

figure.legend(handles=legend_handles, loc="upper center", bbox_to_anchor=(0.5, 1.16), ncol=2, frameon=False)
figure.supxlabel(r"Natural Log Lengthscale, $\mathcal{ln}(\ell)$", fontsize=14)
figure.supylabel("Density", fontsize=14)
figure.suptitle("Pooled Two-Dimensional GP Lengthscales", fontsize=16)

figure.savefig("pooled_age_size_log_lengthscales.pdf", bbox_inches="tight")
figure.savefig("pooled_age_size_log_lengthscales.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
size_lengthscales = pooled_lengthscales[:, 0]
age_lengthscales = pooled_lengthscales[:, 1]

size_prior_log_mean = 0.0
size_prior_log_sd = 1.5
age_prior_log_mean = 2.0
age_prior_log_sd = 1.5

log_grid = np.linspace(-5.5, 5.5, 1000)

size_prior_density = norm.pdf(log_grid, loc=size_prior_log_mean, scale=size_prior_log_sd)
age_prior_density = norm.pdf(log_grid, loc=age_prior_log_mean, scale=age_prior_log_sd)

figure, axes = plt.subplots(1, 2, figsize=(7.2, 3.8), sharex=True)

sns.kdeplot(x=np.log(size_lengthscales), fill=True, cut=0, color="#0072B2", linewidth=0, alpha=0.2, ax=axes[0])
sns.kdeplot(x=np.log(size_lengthscales), cut=0, color="#0072B2", linewidth=1.8, ax=axes[0])
axes[0].plot(log_grid, size_prior_density, color="#333333", linestyle="--", linewidth=1.8)
axes[0].set_title("Size Lengthscale", fontsize=11)

sns.kdeplot(x=np.log(age_lengthscales), fill=True, cut=0, color="#D55E00", linewidth=0, alpha=0.2, ax=axes[1])
sns.kdeplot(x=np.log(age_lengthscales), cut=0, color="#D55E00", linewidth=1.8, ax=axes[1])
axes[1].plot(log_grid, age_prior_density, color="#333333", linestyle="--", linewidth=1.8)
axes[1].set_title("Age Lengthscale", fontsize=11)

for axis in axes:
    axis.set_xlim(-5.5, 5.5)
    axis.set_xlabel("")
    axis.set_ylabel("")
    style_axis(axis)

legend_handles = [
    Line2D([0], [0], color="#0072B2", linewidth=2, label="Size Posterior"),
    Line2D([0], [0], color="#333333", linestyle="--", linewidth=2, label=r"Size Prior: $\sim \mathcal{N}(0,1.5^2)$"),
    Line2D([0], [0], color="#D55E00", linewidth=2, label="Age Posterior"),
    Line2D([0], [0], color="#333333", linestyle="--", linewidth=2, label=r"Age Prior: $\sim \mathcal{N}(2,1.5^2)$"),
]

# Reserve space above the panels
figure.subplots_adjust(left=0.10, right=0.98, bottom=0.15, top=0.67, wspace=0.18)

figure.suptitle("Pooled Two-Dimensional GP Lengthscales", fontsize=14, y=0.98)

figure.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.89),
    ncol=2,
    frameon=False,
    fontsize=9,
)

figure.supxlabel(r"Natural Log Lengthscale, $\ln(\ell)$", fontsize=11)
figure.supylabel("Density", fontsize=11)

figure.savefig("pooled_age_size_log_lengthscales.pdf", bbox_inches="tight")
figure.savefig("pooled_age_size_log_lengthscales.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
from scipy.special import expit

number_simulations = 1000
number_years = soay_data["yr"].nunique()
stop_at = 512
seed = 56

parameter_names = ["growth_int", "growth_var", "growth_ls", "growth_noise", "surv_int", "surv_var", "surv_ls", "repr_int", "repr_var", "repr_ls", "recr_int", "recr_var", "recr_ls", "rcsz_int", "rcsz_var", "rcsz_ls", "rcsz_noise"]

with np.load(samples_path) as saved:
    samples = {name: saved[name] for name in parameter_names}

def draw_gp(inputs, parameters, rng, jitter=5e-4):
    inputs = np.asarray(inputs, dtype=float)
    inputs = inputs[:, None] if inputs.ndim == 1 else inputs
    unique_inputs, inverse = np.unique(inputs, axis=0, return_inverse=True)
    differences = (unique_inputs[:, None, :] - unique_inputs[None, :, :]) / parameters["lengthscale"]
    covariance = parameters["variance"] * np.exp(-0.5 * np.sum(differences**2, axis=-1))
    covariance.flat[::len(unique_inputs) + 1] += jitter
    values = parameters["intercept"] + np.linalg.cholesky(covariance) @ rng.normal(size=len(unique_inputs))
    return values[inverse], len(unique_inputs)

def draw_parameters(rng):
    parameters = {}
    for name in ["growth", "surv", "repr", "recr", "rcsz"]:
        index = rng.integers(len(samples[f"{name}_int"]))
        parameters[name] = {"intercept": samples[f"{name}_int"][index], "variance": samples[f"{name}_var"][index], "lengthscale": samples[f"{name}_ls"][index]}
        if f"{name}_noise" in samples:
            parameters[name]["noise"] = samples[f"{name}_noise"][index]
    return parameters

def simulate_year(size, age, parameters, rng):
    records = []
    age_size = np.column_stack([size, age])

    survival_value, locations = draw_gp(age_size, parameters["surv"], rng)
    records.append(("Survival", locations))
    survivor = rng.binomial(1, expit(survival_value)).astype(bool)

    next_size = np.full(len(size), np.nan)
    if survivor.any():
        growth_mean, locations = draw_gp(age_size[survivor], parameters["growth"], rng)
        records.append(("Growth", locations))
        next_size[survivor] = growth_mean + rng.normal(0, parameters["growth"]["noise"], survivor.sum())

    eligible = survivor & (age >= 1)
    breeder = np.zeros(len(size), dtype=bool)
    if eligible.any():
        reproduction_value, locations = draw_gp(age_size[eligible], parameters["repr"], rng)
        records.append(("Reproduction", locations))
        breeder[eligible] = rng.binomial(1, expit(reproduction_value)).astype(bool)

    female = breeder & rng.binomial(1, 0.5, len(size)).astype(bool)
    recruit = np.zeros(len(size), dtype=bool)
    if female.any():
        recruitment_value, _ = draw_gp(age[female], parameters["recr"], rng)
        recruit[female] = rng.binomial(1, expit(recruitment_value)).astype(bool)

    recruit_size = np.empty(0)
    if recruit.any():
        recruit_mean, _ = draw_gp(size[recruit], parameters["rcsz"], rng)
        recruit_size = recruit_mean + rng.normal(0, parameters["rcsz"]["noise"], recruit.sum())

    next_population_size = np.concatenate([recruit_size, next_size[survivor]])
    next_population_age = np.concatenate([np.zeros(len(recruit_size), dtype=int), np.minimum(age[survivor] + 1, 8)])

    return next_population_size, next_population_age, records

first_year = soay_data["yr"].min()
initial_population = soay_data.loc[soay_data["yr"].eq(first_year), ["z", "age"]].dropna()

call_records = []
simulation_records = []

for simulation in range(number_simulations):
    rng = np.random.default_rng(seed + simulation)
    parameters = draw_parameters(rng)
    size = initial_population["z"].to_numpy(float).copy()
    age = initial_population["age"].to_numpy(int).copy()
    maximum_locations = 0
    capped = False

    for year in range(number_years):
        if len(size) <= 1:
            break

        current_locations = len(np.unique(np.column_stack([size, age]), axis=0))
        maximum_locations = max(maximum_locations, current_locations)

        if current_locations > stop_at:
            capped = True
            break

        size, age, year_records = simulate_year(size, age, parameters, rng)

        for vital_rate, locations in year_records:
            call_records.append({"Simulation": simulation, "Year": year + 1, "Vital Rate": vital_rate, "Locations": locations})
            maximum_locations = max(maximum_locations, locations)

    simulation_records.append({"Simulation": simulation, "Maximum Locations": maximum_locations, "Capped": capped})

call_counts = pd.DataFrame(call_records)
simulation_counts = pd.DataFrame(simulation_records)

print(call_counts.groupby("Vital Rate")["Locations"].describe(percentiles=[0.50, 0.90, 0.95, 0.99]).round(1))
print("\nMaximum Locations Per Simulation:")
print(simulation_counts["Maximum Locations"].describe(percentiles=[0.50, 0.90, 0.95, 0.99]).round(1))
print(f"\nCapped Simulations: {simulation_counts['Capped'].sum()} / {number_simulations}")
print("Smallest Nonempty Call:", call_counts["Locations"].min())

observed_maximum = soay_data.dropna(subset=["z", "age"]).groupby("yr")[["z", "age"]].apply(lambda values: len(values.drop_duplicates())).max()
maximum_to_cover = max(observed_maximum, simulation_counts["Maximum Locations"].max())
recommended_maximum = 2 ** int(np.ceil(np.log2(1.25 * maximum_to_cover)))

print("Observed Maximum:", observed_maximum)
print("Simulated Maximum:", simulation_counts["Maximum Locations"].max())
print("Suggested Maximum With 25% Margin:", recommended_maximum)

if simulation_counts["Capped"].any():
    print("Some simulations reached the safety limit; increase stop_at before making the final choice.")